* 本例展示如何在alpha-mind中使用机器学习模型

* 请在环境变量中设置`DB_URI`指向数据库

In [ ]:
import asyncio


## 数据配置
------------

In [ ]:
import datetime as dt
import numpy as np


我们使用当期的`["EMA5D", "EMV6D"]`因子，来尝试预测未来大概一个月以后的收益。

* 训练的股票池为`hs300`;；
* 因子都经过中性化以及标准化等预处理；
* 对于线性模型，我们以10个工作日为一个时间间隔，用过去8期的数据作为训练用特征。

In [ ]:
import pyarrow as pa


## 模型预测
-----------

In [ ]:
import pyarrow.compute as pc

rng = np.random.default_rng(42)


In [ ]:
n_dates = 6


## 模型对比 （线性回归模型 v.s. Naive - 常数线性模型）
------------------

In [ ]:
n_assets = 12


In [ ]:
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


In [ ]:
n_rows = n_dates * n_assets


## 模型时间序列
-------------------------

通过比较在测试集的结果，我们观察如下两个模型的表现：

* Naive Model：简单的使用因子上期值作为当期值的预测；
* 线性回归模型：利用过去四期的因子值回归后得到模型，然后用这个模型预测当期值；

In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.model import LinearRegression, load_model


In [ ]:
train = panel.filter(pc.less(panel["trade_date"], dates[-1]))


在这个例子中，线性回归模型的IC值略微高于Naive模型。

## 回测（ simple long short strategy）
--------------------------

In [ ]:
test = panel.filter(pc.equal(panel["trade_date"], dates[-1]))


In [ ]:
model = LinearRegression(features=["factor", "value"], fit_intercept=True)
model.fit(train, train["dx"].to_numpy(zero_copy_only=False))


In [ ]:
    ## naive model
    ## regression model
predictions = pa.table({
    "trade_date": test["trade_date"],
    "code": test["code"],
    "prediction": model.predict(test),
})


In [ ]:
restored = load_model(model.save())
np.testing.assert_allclose(restored.predict(test), predictions["prediction"].to_pylist())


In [ ]:
predictions
